# VK-LSVD → inter.json + content_embeddings.pkl

Аналог `notebooks/DatasetProcessing.ipynb` для VK-LSVD. На выходе:
* `../data/VK/inter.json` — интеракции в формате `{str(user_id): [item_id, ...]}`
* `../data/VK/content_embeddings.pkl` — `{'item_id': [...], 'embedding': [...]}` со встроенными VK эмбедами.

Шаги: скачивание сабсэмпла `ur0.01_ir0.01` через `hf download` (как в `LsvdDownload.ipynb`), фильтрация по `timespent > 15`, пересечение с эмбедами, **итеративный Core-5 фильтр**, ремап `user_id`/`item_id` в плотный 0-индекс, группировка по юзеру.


In [21]:
%pip install polars==1.36.1
%pip install numpy
%pip install pyarrow
%pip install huggingface_hub

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [22]:
import json
import os
import pickle
from collections import defaultdict

import numpy as np
import polars as pl

## Конфигурация

Пути и параметры. На A100-сервере оставляем `/home/jovyan/...`. На других машинах — переопределяем `DATASET_PATH`.

In [23]:
SUBSAMPLE = 'ur0.01_ir0.01'
DATASET_PATH = '/home/jupyter/project/tiger-cf/vk_lsvd'
POSITIVE_EVENT_TIMESPENT = 15
CORE_K = 5

INTERACTIONS_OUTPUT_PATH = '../data/VK/inter.json'
EMBEDDINGS_OUTPUT_PATH = '../data/VK/content_embeddings.pkl'

os.makedirs(os.path.dirname(INTERACTIONS_OUTPUT_PATH), exist_ok=True)

Тестовой скачивание

In [24]:
os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

In [25]:
import os
from huggingface_hub import hf_hub_download

os.environ['HF_ENDPOINT'] = 'https://huggingface.co'

metadata_files = [
    'metadata/users_metadata.parquet',
    'metadata/items_metadata.parquet',
    'metadata/item_embeddings.npz',
]
for file in metadata_files:
    hf_hub_download(
        repo_id='deepvk/VK-LSVD',
        repo_type='dataset',
        filename=file,
        local_dir=DATASET_PATH,
    )
    print(f'downloaded: {file}')

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 4c611160-3945-4406-ac61-2296063f00ed)')' thrown while requesting HEAD https://huggingface.co/datasets/deepvk/VK-LSVD/resolve/main/metadata/users_metadata.parquet
Retrying in 1s [Retry 1/5].


downloaded: metadata/users_metadata.parquet
downloaded: metadata/items_metadata.parquet
downloaded: metadata/item_embeddings.npz


In [26]:
from huggingface_hub import list_repo_files

subsample_files = [
    f for f in list_repo_files('deepvk/VK-LSVD', repo_type='dataset')
    if f.startswith(f'subsamples/{SUBSAMPLE}/')
]
print(f'files to download: {len(subsample_files)}')
for file in subsample_files:
    hf_hub_download(
        repo_id='deepvk/VK-LSVD',
        repo_type='dataset',
        filename=file,
        local_dir=DATASET_PATH,
    )
    print(f'downloaded: {file}')

files to download: 27
downloaded: subsamples/ur0.01_ir0.01/test/week_26.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_00.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_01.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_02.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_03.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_04.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_05.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_06.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_07.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_08.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_09.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_10.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_11.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_12.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_13.parquet
downloaded: subsamples/ur0.01_ir0.01/train/week_14.parquet
downloaded: subsamples/ur0.01_ir0.0

## Скачивание (выполнять один раз)

Ячейки ниже — копия `LsvdDownload.ipynb`. Если уже скачано в `DATASET_PATH`, можно пропустить.

In [6]:
!HF_ENDPOINT="http://huggingface.co" hf download deepvk/VK-LSVD --repo-type dataset --include "metadata/*" --local-dir {DATASET_PATH}

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/home/jupyter/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py", line 1568, in _get_metadata_or_catch_error
    raise FileMetadataError(
huggingface_hub.errors.FileMetadataError: Distant resource does not seem to be on huggingface.co. It is possible that a configuration issue prevents you from downloading resources from https://huggingface.co. Please check your firewall and proxy settings and make sure your SSL certificates are updated.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/jupyter/.local/bin/hf", line 7, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/jupyter/.local/lib/python3.11/site-packages/huggingface_hub/cli/hf.py", line 59, in main
    service.run()
  File "/home/jupyter/.local/lib/python3.11/site-packages/huggingface_hub/cli/download.py", line 136, in run
    p

Exception: Process exited with code 1

In [ ]:
!HF_ENDPOINT="http://huggingface.co" hf download deepvk/VK-LSVD --repo-type dataset --include "subsamples/{SUBSAMPLE}/*" --local-dir {DATASET_PATH}

## Чтение интеракций (все 25 недель)

Конкатенируем base+gap+val+test (как договорились — leave-one-out внутри TIGER делается по позиции, поэтому в `inter.json` идёт вся история).

In [27]:
ALL_WEEKS = list(range(25))
all_files = [f'subsamples/{SUBSAMPLE}/train/week_{i:02}.parquet' for i in ALL_WEEKS]
len(all_files)

25

In [28]:
def get_parquet_interactions(data_files, positive_event_timespent):
    df = pl.concat([pl.scan_parquet(f'{DATASET_PATH}/{f}') for f in data_files]).collect()
    df = df.with_row_index('original_order')
    df = df.filter(pl.col('timespent') > positive_event_timespent)
    return df

all_inter = get_parquet_interactions(all_files, POSITIVE_EVENT_TIMESPENT)
print('после timespent-фильтра:', all_inter.shape)

после timespent-фильтра: (1417165, 13)


## Пересечение с эмбедами

Выбрасываем интеракции с айтемами, у которых нет эмбеддинга в `metadata/item_embeddings.npz`.

In [29]:
emb_npz = np.load(f'{DATASET_PATH}/metadata/item_embeddings.npz')
emb_item_ids = emb_npz['item_id']
emb_vectors = emb_npz['embedding']
print('эмбедов всего:', emb_item_ids.shape, emb_vectors.shape)

эмбедов всего: (19627601,) (19627601, 64)


In [30]:
items_with_emb = pl.DataFrame({'item_id': emb_item_ids})
filtered_df = all_inter.join(items_with_emb, on='item_id', how='inner')
print('после пересечения с эмбедами:', filtered_df.shape)

после пересечения с эмбедами: (1417165, 13)


## Core-5 фильтрация

Итеративно выкидываем юзеров и айтемы с <5 интеракций до сходимости (как в `DatasetProcessing.ipynb`).

In [31]:
is_changed = True
iteration = 0
while is_changed:
    iteration += 1
    user_counts = filtered_df.group_by('user_id').agg(pl.len().alias('user_count'))
    item_counts = filtered_df.group_by('item_id').agg(pl.len().alias('item_count'))

    good_users = user_counts.filter(pl.col('user_count') >= CORE_K).select('user_id')
    good_items = item_counts.filter(pl.col('item_count') >= CORE_K).select('item_id')

    old_size = len(filtered_df)
    new_df = filtered_df.join(good_users, on='user_id', how='inner')
    new_df = new_df.join(good_items, on='item_id', how='inner')
    new_size = len(new_df)

    filtered_df = new_df
    is_changed = old_size != new_size
    print(f'iter {iteration}: {old_size} -> {new_size}')

print('финал после Core-5:', filtered_df.shape)

iter 1: 1417165 -> 1291348
iter 2: 1291348 -> 1285003
iter 3: 1285003 -> 1284408
iter 4: 1284408 -> 1284352
iter 5: 1284352 -> 1284340
iter 6: 1284340 -> 1284340
финал после Core-5: (1284340, 13)


## Ремап user_id и item_id в 0-indexed dense

Сохраняем порядок появления — это даст компактные id.

In [32]:
unique_users = filtered_df['user_id'].unique(maintain_order=True).to_list()
user_ids_mapping = {value: i for i, value in enumerate(unique_users)}

unique_items = filtered_df['item_id'].unique(maintain_order=True).to_list()
item_ids_mapping = {value: i for i, value in enumerate(unique_items)}

num_users = len(user_ids_mapping)
num_items = len(item_ids_mapping)
print('num_users:', num_users, 'num_items:', num_items)

num_users: 49691 num_items: 21928


In [33]:
filtered_df = filtered_df.with_columns([
    pl.col('user_id').replace_strict(user_ids_mapping).alias('user_id'),
    pl.col('item_id').replace_strict(item_ids_mapping).alias('item_id'),
])
filtered_df.head()

original_order,user_id,item_id,place,platform,agent,timespent,like,dislike,share,bookmark,click_on_author,open_comments
u32,i64,i64,u8,u8,u8,u8,bool,bool,bool,bool,bool,bool
1380444,0,0,1,0,0,16,false,false,false,false,false,false
1380465,1,0,1,0,0,95,false,false,false,false,false,false
1383779,2,0,1,0,0,60,false,false,false,false,false,false
1386069,3,0,1,1,1,62,false,false,false,false,false,false
1388804,4,0,1,0,0,61,false,false,false,false,false,false


## Группировка по юзеру и сериализация inter.json

Сортируем по `original_order` (это и есть таймстемп в LSVD), чтобы в листе айтемов сохранялся хронологический порядок.

In [34]:
filtered_df = filtered_df.sort(['user_id', 'original_order'])
grouped = (
    filtered_df
    .group_by('user_id', maintain_order=True)
    .agg(pl.col('item_id'))
)
grouped.head()

user_id,item_id
i64,list[i64]
0,"[20501, 158, … 4818]"
1,"[4786, 18700, … 9850]"
2,"[21899, 12795, … 1528]"
3,"[7408, 14140, … 17373]"
4,"[20404, 3306, … 15745]"


In [35]:
json_data = {}
for user_id, item_list in grouped.iter_rows():
    json_data[int(user_id)] = list(map(int, item_list))

# sanity: Core-5 после ремапа должен сохраниться
assert all(len(v) >= CORE_K for v in json_data.values()), 'Core-5 broken после ремапа'
assert max(max(v) for v in json_data.values()) == num_items - 1
assert min(min(v) for v in json_data.values()) == 0

with open(INTERACTIONS_OUTPUT_PATH, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f'inter.json: {len(json_data)} юзеров, диапазон item_id [0, {num_items - 1}]')

inter.json: 49691 юзеров, диапазон item_id [0, 21927]


## Сохранение content_embeddings.pkl

Переупорядочиваем `item_embeddings.npz` под новый item_id mapping. Формат — как в Amazon-пайплайне: `dict('item_id' -> list[int], 'embedding' -> list[np.ndarray(D,) float32])`.

In [36]:
old_to_new = item_ids_mapping  # {old_item_id: new_item_id} (подмножество от npz)

# индекс эмбедов по старому item_id
old_id_to_pos = {int(old): i for i, old in enumerate(emb_item_ids)}

new_item_ids = list(range(num_items))
new_embeddings = np.zeros((num_items, emb_vectors.shape[1]), dtype=np.float32)
for old_id, new_id in old_to_new.items():
    new_embeddings[new_id] = emb_vectors[old_id_to_pos[int(old_id)]].astype(np.float32)

print('new_embeddings.shape:', new_embeddings.shape)
print('non-zero rows:', int(np.any(new_embeddings != 0, axis=1).sum()))

new_embeddings.shape: (21928, 64)
non-zero rows: 21928


In [37]:
out = {
    'item_id': new_item_ids,
    'embedding': [new_embeddings[i] for i in range(num_items)],
}
with open(EMBEDDINGS_OUTPUT_PATH, 'wb') as f:
    pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'content_embeddings.pkl сохранён: {EMBEDDINGS_OUTPUT_PATH}')
print(f'  num_items: {len(out["item_id"])}, D: {out["embedding"][0].shape[0]}')

content_embeddings.pkl сохранён: ../data/VK/content_embeddings.pkl
  num_items: 21928, D: 64


## Финальные sanity-чеки

In [38]:
with open(INTERACTIONS_OUTPUT_PATH) as f:
    inter = json.load(f)
with open(EMBEDDINGS_OUTPUT_PATH, 'rb') as f:
    emb = pickle.load(f)

assert set(map(int, inter.keys())) == set(range(len(inter)))
assert emb['item_id'] == list(range(len(emb['item_id'])))
assert max(max(v) for v in inter.values()) == len(emb['item_id']) - 1
print('OK:', len(inter), 'юзеров,', len(emb['item_id']), 'айтемов,', emb['embedding'][0].shape, 'эмбед')

OK: 49691 юзеров, 21928 айтемов, (64,) эмбед
